# Silver - Canais Comerciais

Limpeza do cadastro de canais de vendas (sales channels).

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
sistema = 'case'
table_name = 'comercial_canais'
input_path = f"{var_bronze}/{sistema}/{table_name}/data"
output_path_data = f"{var_silver}/{sistema}/{table_name}/data"
table_name_schema = f'{var_environment}.{var_silver_schema}.{sistema}_{table_name}'

In [ ]:
from pyspark.sql.functions import col, trim

df_bronze = spark.read.format("delta").load(input_path)

df_clean = (
    df_bronze
    .select(
        col("canal_id").cast("integer").alias("id_canal"),
        trim(col("canal")).cast("string").alias("nome_canal")
    )
    .filter(col("id_canal").isNotNull())
    .dropDuplicates(["id_canal"])
)

In [ ]:
process_data(
    df_write=df_clean,
    tipo_carga='full',
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    chave_clusterby=['nome_canal'],
    chave_upsert='id_canal'
)